In [1]:
import sys
import torch
sys.path.insert(0,"../src")

from cnn_project.train import train_model
from cnn_project.model import EuroSATModel
from cnn_project.evaluate import evaluate_model
from cnn_project.data import create_data_loaders
from pathlib import Path
import matplotlib.pyplot as plt
import optuna

/home/h171138/EuroSAT-Model/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("The model is using CUDA")
else:
    device = torch.device("cpu")
    print("The model is using CPU")

torch.manual_seed(42)
model = EuroSATModel().to(device)
train_loader, val_loader, test_loader= create_data_loaders(batch_size=64)

The model is using CUDA


Just checking the optimal number of workers for dataloaders

In [3]:
import time
from torch.utils.data import DataLoader

images_to_decode = train_loader.dataset

for workers in [0, 2, 2, 4, 4, 8, 8, 10, 10, 12, 12, 14, 14, 16, 16]:
    loader = DataLoader(
        images_to_decode,
        batch_size=train_loader.batch_size,
        shuffle=False,
        num_workers=workers,
        persistent_workers=(workers > 0),
        prefetch_factor=2 if workers > 0 else None,
    )

    # First pass: start workers and warm the file cache.
    for images, labels in loader:
        pass

    # Second pass: measure images actually processed.
    count = 0
    start = time.perf_counter()

    for images, labels in loader:
        count += images.size(0)

    elapsed = time.perf_counter() - start
    print(
        f"{workers} workers: {count} images in {elapsed:.2f} s "
        f"({count / elapsed:.0f} images/s)"
    )

0 workers: 21600 images in 6.15 s (3511 images/s)
2 workers: 21600 images in 3.72 s (5801 images/s)
2 workers: 21600 images in 3.86 s (5602 images/s)
4 workers: 21600 images in 2.60 s (8303 images/s)
4 workers: 21600 images in 2.38 s (9090 images/s)
8 workers: 21600 images in 2.33 s (9280 images/s)
8 workers: 21600 images in 2.25 s (9598 images/s)
10 workers: 21600 images in 2.09 s (10343 images/s)
10 workers: 21600 images in 2.17 s (9968 images/s)
12 workers: 21600 images in 1.92 s (11249 images/s)
12 workers: 21600 images in 1.95 s (11078 images/s)
14 workers: 21600 images in 1.77 s (12181 images/s)
14 workers: 21600 images in 1.90 s (11362 images/s)
16 workers: 21600 images in 1.78 s (12154 images/s)
16 workers: 21600 images in 1.70 s (12676 images/s)
